# LiDAR target distribution

The global histogram of every valid LiDAR residual value across the paired
Tuktoyaktuk patches -- the counterpart of the reference dissertation's
"Distribution of LiDAR Values" figure. No model, no GPU; reads the patch
files and bins the pixels.

Two things it establishes for Section 3.1: that the target is centred on
zero (the RANSAC detrending and per-patch demeaning have done their job),
and how heavy the tails are relative to the 0.172 m standard deviation the
text quotes.

In [ ]:
import json
from pathlib import Path
import numpy as np
import rasterio
import matplotlib.pyplot as plt

WORKING_REPO = Path('/cs/student/project_msc/2025/aibh/jiayiche')
LIDAR_DIR = WORKING_REPO / 'input_data' / 'lidar_patches_tuk_tessa'
S1_DIR = WORKING_REPO / 'input_data' / 's1_patches_tuk_pcrtc'
OUTPUT_DIR = WORKING_REPO / 's1_training_outputs'

lidar_ids = {p.stem.split('_')[-1] for p in LIDAR_DIR.glob('lidar_patch_*.tif')}
s1_ids = {p.name.split('_')[-1] for p in S1_DIR.glob('s1_patch_*') if p.is_dir()}
paired = sorted(lidar_ids & s1_ids)
print(f'{len(paired)} paired patches')

# Per-patch demeaned values, exactly as the dataset class produces them.
vals, patch_std = [], []
for pid in paired:
    with rasterio.open(LIDAR_DIR / f'lidar_patch_{pid}.tif') as src:
        raw = src.read().astype(np.float32)
    z = raw[0]; mask = (raw[1] > 0.5) if raw.shape[0] > 1 else np.isfinite(z)
    z = np.nan_to_num(z, nan=0.0, posinf=0.0, neginf=0.0)
    v = z[mask]
    if v.size == 0: continue
    v = v - v.mean()
    vals.append(v); patch_std.append(v.std())
vals = np.concatenate(vals); patch_std = np.array(patch_std)

print(f'valid pixels      : {vals.size:,}')
print(f'global mean       : {vals.mean():+.4f} m   (should be ~0)')
print(f'global std        : {vals.std():.4f} m')
print(f'min / max         : {vals.min():+.3f} / {vals.max():+.3f} m')
for q in [0.1, 1, 5, 50, 95, 99, 99.9]:
    print(f'  {q:5.1f}th pct    : {np.percentile(vals, q):+.4f} m')
print(f'mean per-patch std: {patch_std.mean():.4f} m   median {np.median(patch_std):.4f} m')
print(f'pixels beyond +-1 m: {100*np.mean(np.abs(vals) > 1):.3f}%')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

ax = axes[0]
ax.hist(vals, bins=400, range=(-2, 2), density=True, color='#1D4E63', alpha=0.85)
ax.set_yscale('log')
ax.set_xlabel('LiDAR residual after per-patch demeaning (m)')
ax.set_ylabel('density (log)')
ax.set_title(f'All valid pixels, {len(paired)} patches, n = {vals.size:,}', fontsize=10)
ax.axvline(0, color='k', lw=0.6)
for s in (-vals.std(), vals.std()):
    ax.axvline(s, color='#C2561F', lw=0.8, ls='--')
ax.text(vals.std(), ax.get_ylim()[1]*0.5, f'  \u00b11 s.d. = {vals.std():.3f} m', color='#C2561F', fontsize=8, va='top')

ax = axes[1]
ax.hist(patch_std, bins=60, color='#1D4E63', alpha=0.85)
ax.axvline(patch_std.mean(), color='#C2561F', lw=1, ls='--', label=f'mean {patch_std.mean():.3f} m')
ax.axvline(np.median(patch_std), color='k', lw=1, ls=':', label=f'median {np.median(patch_std):.3f} m')
ax.set_xlabel('Within-patch standard deviation (m)')
ax.set_ylabel('patches')
ax.set_title('Per-patch roughness', fontsize=10)
ax.legend(fontsize=8)

plt.tight_layout()
out = OUTPUT_DIR / 'fig_lidar_distribution.png'
plt.savefig(out, dpi=200, bbox_inches='tight', facecolor='white')
print('Saved:', out)
plt.show()

json.dump({'n_pixels': int(vals.size), 'global_std_m': float(vals.std()),
           'min_m': float(vals.min()), 'max_m': float(vals.max()),
           'mean_patch_std_m': float(patch_std.mean()), 'median_patch_std_m': float(np.median(patch_std)),
           'pct_beyond_1m': float(100*np.mean(np.abs(vals) > 1))},
          (OUTPUT_DIR / 'fig_lidar_distribution_stats.json').open('w'), indent=2)